In [ ]:
%run Setting_Env_Variables_p2.py
google_project_id = %env GOOGLE_CLOUD_PROJECT
bucket = os.getenv("WORKSPACE_BUCKET")

import os
import subprocess
import pandas as pd
from datetime import datetime

import hail as hl
hl.init(gcs_requester_pays_configuration=google_project_id) #, log=f'{bucket}/hail_logs')
hl.default_reference(new_default_reference = "GRCh38")


In [ ]:
auxiliary_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux"
print(f'aux path: {auxiliary_path}')

vat_path = f'{auxiliary_path}/vat/*.gz'
print(f'vat path: {vat_path}')

clinvar_ht_path = 'gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/clinvar/multiMT/hail.mt'
print(f'clinvar ht path: {clinvar_ht_path}')

vds_path = 'gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/vds/hail.vds'
print(f'vds path: {vds_path}')

flagged_samples_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/qc/flagged_samples.tsv"
print(f'flagged samples path: {flagged_samples_path}')


# mt_wgs_path = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/multiMT/hail.mt"
# flagged_samples = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/aux/qc/flagged_samples.tsv"
# array_path = "gs://vwb-aou-datasets-controlled/v9/microarray/hail.mt"
# # this path is in the official ct v8 fw
# bed_path = "gs://aou-tutorial-notebooks-wb-sunny-radish-6214/genomic_test_data/random_sites_GRCh38.txt"

In [ ]:
%%time

types={'clinvar_classification': hl.tarray(hl.tstr), 
       'position': hl.tint, 
       'omim_phenotypes_id':hl.tarray(hl.tint), 
       'omim_phenotypes_name':hl.tarray(hl.tstr),
       'clinvar_phenotype':hl.tarray(hl.tstr)
      }



vat_table = hl.import_table(vat_path, force=True, quote='"', delimiter="\t", force_bgz=True, types=types) # impute=True)
vat_table = vat_table.add_index(name='id')

parse by demographic

Get all LMNA associated variants for control groups

In [ ]:
gene = 'LMNA'
ens_id = 'ENSG00000160789'

lmna_vat = vat_table.filter(vat_table["gene_id"]==ens_id)

In [ ]:
# filt_vat.write(f'{bucket}/hail_files/lmna_vat.ht', overwrite=True)


In [ ]:
filt_vat.describe()

In [ ]:
filtered_ht = filt_vat.filter(hl.any(
    # filt_vat.consequence.contains('intron_variant'),
    # filt_vat.consequence.contains('non_coding_transcript_variant'),
    filt_vat.clinvar_classification.contains('pathogenic'),
    filt_vat.clinvar_classification.contains('uncertain significance'),
    filt_vat.clinvar_classification.contains('likely pathogenic'),
))

In [ ]:
# filtered_ht.write(f'{bucket}/hail_files/lmna_vat.ht', overwrite=True)


In [ ]:
# print(f'{gene} VAT length = {filtered_ht.count()}')

# get variants within 1 Mb of LMNA

In [ ]:
ct = hl.read_matrix_table(clinvar_ht_path)

In [ ]:
ct.describe()

In [ ]:
%%time
ct.count()

# vds

In [ ]:
%%time
vds = hl.vds.read_vds(vds_path)

In [ ]:
# only work with data from chromosome 1 (location of LMNA)
vds = hl.vds.filter_chromosomes(vds, keep= ["chr1"])

In [ ]:
# change coordinates to the variant you are interested in
#chr1:156082572-156140081
start = 156082572
stop = 156140081
interval = f'chr1:{start-1000000}-{stop+1000000}'
print(interval)
intervals = [interval]

In [ ]:
%%time
vds = hl.vds.filter_intervals(
    vds,
    [hl.parse_locus_interval(x,)
     for x in intervals])

In [ ]:
%%time
vds.variant_data.count()

# count() (971624, 535662)

The field sample_id contains the person_ids, so we set the field sample_id as key while reading the table.

In [ ]:
flagged_samples = hl.import_table(flagged_samples_path, key = "s")

In [ ]:
vds = hl.vds.filter_samples(vds, flagged_samples, keep = False, remove_dead_alleles = True)

In [ ]:
# %%time
# vds.variant_data.count()

In [ ]:
vds.variant_data.describe()